<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/clean_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)

print("folder ready")

Mounted at /content/drive
folder ready


In [11]:
%%writefile /content/drive/MyDrive/ml_project/clean_data.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

TARGET_DIR = "/content/drive/MyDrive/ml_project"
INPUT_PICKLE = os.path.join(TARGET_DIR, "df_date.pkl")
METHOD_FILE = os.path.join(TARGET_DIR, "clean_method.txt")
OUTPUT_PICKLE = os.path.join(TARGET_DIR, "df_clean.pkl")

# ---------------------------------------------------------
# clean_data_ui – per-column settings
# ---------------------------------------------------------
def clean_data_ui():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_date.pkl not found")

    df = pd.read_pickle(INPUT_PICKLE)
    missing = df.isna().sum()

    # list of labels: "column (X missing)"
    labels = []
    dropdowns = []

    for col in df.columns:
        miss = int(missing[col])
        label = widgets.Label(f"{col}: {miss} missing")
        dd = widgets.Dropdown(
            options=["drop", "zero", "ffill", "mean", "none"],
            description=col,
        )
        labels.append(label)
        dropdowns.append(dd)

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        with open(METHOD_FILE, "w") as f:
            for dd in dropdowns:
                col = dd.description
                method = dd.value
                f.write(col + ":" + method + "\n")

        with out:
            print("saved:", METHOD_FILE)
            print("methods saved for all columns")

    btn.on_click(on_click)

    display(widgets.VBox(labels + dropdowns + [btn, out]))


# ---------------------------------------------------------
# clean_data – apply per-column cleaning
# ---------------------------------------------------------
def clean_data():

    if not os.path.exists(INPUT_PICKLE):
        raise FileNotFoundError("df_date.pkl not found")

    if not os.path.exists(METHOD_FILE):
        raise FileNotFoundError("clean_method.txt not found")

    df = pd.read_pickle(INPUT_PICKLE)

    # read methods per column
    methods = {}
    with open(METHOD_FILE, "r") as f:
        for line in f.readlines():
            col, method = line.strip().split(":")
            methods[col] = method

    # apply method per column
    for col, method in methods.items():

        if method == "drop":
            df = df[df[col].notna()]

        elif method == "zero":
            df[col] = df[col].fillna(0)

        elif method == "ffill":
            df[col] = df[col].fillna(method="ffill")

        elif method == "mean":
            if pd.api.types.is_numeric_dtype(df[col]):
                mean_val = df[col].mean()
                df[col] = df[col].fillna(mean_val)
            else:
                df[col] = df[col].fillna(method="ffill")

        elif method == "none":
            pass

    df.to_pickle(OUTPUT_PICKLE)

    print("saved:", OUTPUT_PICKLE)
    print("applied per-column cleaning methods")

    return df


Overwriting /content/drive/MyDrive/ml_project/clean_data.py


In [13]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/ml_project')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from clean_data import clean_data_ui
clean_data_ui()